# Expected 2027 performance

**Brief: Expected 2027 performance + range.**

- Demand trend: reviews per month (growth, seasonality, World Cup spike in May–Jun 2026)
- Forward bookings: calendar unavailability by month (booked OR blocked)
- Range: `sens`

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import config
T = config.TABLES   # run `python run.py` from the repo root first if outputs/ is empty

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.2f}".format)
%matplotlib inline

## Data
- `reviews`: one row per review (listing_id, date)
- `cal`: listing × night for the next 365 days (available yes/no, no prices)
- `sens`: low / base / high / 2-months-offline

### From 01 (keep in mind)
- **Occupancy & revenue** = Inside Airbnb's estimates (`estimated_occupancy_l365d`, `estimated_revenue_l365d`), 12 months to 15 June 2026. Estimates, not real bookings.
- **Price** = one night's quote around June–July 2026 (World Cup period), so revenue may be a bit high.
- **`is_active`** = reviewed in the last 12 months. Only active listings are used for segments.
- Cleaned data: 6,509 → 4,447 listings (no price, price outside $30–$2,000, 90+ night minimum removed: BC/Vancouver short-term = under 90 nights).

In [ ]:
reviews = pd.read_csv(config.RAW / "reviews.csv.gz", usecols=["listing_id", "date"])  # no reviewer names
reviews["date"] = pd.to_datetime(reviews["date"], format="%Y-%m-%d")
cal = pd.read_csv(config.RAW / "calendar.csv.gz", usecols=["listing_id", "date", "available"])
cal["date"] = pd.to_datetime(cal["date"], format="%Y-%m-%d")
cal["unavail"] = cal["available"].eq("f")
sens = pd.read_csv(T / "sensitivity.csv")

## Quick look

In [ ]:
rev_monthly = reviews.set_index("date").resample("MS").size().rename("reviews")
rev_monthly["2022":].plot(figsize=(10, 3), title="Reviews per month (proxy for stays)"); plt.show()

In [ ]:
cal.groupby(cal["date"].dt.to_period("M"))["unavail"].mean().plot(kind="bar", figsize=(10, 3),
    title="Share of nights unavailable, by month ahead"); plt.show()

In [ ]:
sens

## Analysis
_Start here._